# AdaLux — Training & Evaluation

Questo notebook struttura in un unico flusso riproducibile il **training** e
la **valutazione** di **AdaLux**, l'ambiente Gymnasium custom (`lighting_env.AdaLux`)
che simula il controllo adattivo dell'illuminazione (tapparelle + LED),
organizzato per rispondere a tre domande di ricerca:

- **Q1** — Un agente RL supera un controllore a soglie fisse?
  *Metodo*: training SAC vs baseline a soglie fisse sullo stesso ambiente.
  *Valutazione*: errore medio di lux ed energia consumata su un set di
  episodi di test.
- **Q2** — Quale algoritmo converge meglio ed evita ottimi locali?
  *Metodo*: training PPO e SAC a parità di step.
  *Valutazione*: curve di reward a confronto + osservazione qualitativa del
  comportamento appreso.
- **Q3** — La policy resta affidabile fuori distribuzione (OOD)?
  *Metodo*: test su episodi con condizioni estreme non viste in training.
  *Valutazione*: confronto errore di lux/energia normale vs estremo, più
  osservazione di movimenti bruschi degli attuatori.

Struttura del notebook:

0. Setup
1. Ambiente e funzioni di valutazione comuni
2. **Q1** — SAC vs baseline a soglie fisse
3. **Q2** — PPO vs SAC a parità di step
4. **Q3** — Robustezza fuori distribuzione
5. Conclusioni


## 0. Setup

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

%matplotlib inline

from gymnasium.wrappers import TimeLimit
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy

# Il progetto vive nella cartella corrente: lighting_env.py, baseline_controller.py
sys.path.insert(0, os.getcwd())
from lighting_env import AdaLux
from baseline_controller import ThresholdController


class TqdmProgressCallback(BaseCallback):
    '''Barra di avanzamento per model.learn() basata su tqdm.auto invece che
    su model.learn(progress_bar=True) di Stable-Baselines3 (che internamente
    usa tqdm.rich): quest'ultima è nota per "bloccarsi" visivamente in vari
    ambienti Jupyter/VS Code (il training prosegue comunque in background,
    ma la barra non si aggiorna più a schermo). tqdm.auto è più robusta:
    usa un widget ipywidgets se disponibile, altrimenti una barra testuale
    che si aggiorna correttamente riga per riga.'''

    def __init__(self, total_timesteps, desc="Training"):
        super().__init__()
        self.total_timesteps = total_timesteps
        self.desc = desc
        self.pbar = None

    def _on_training_start(self):
        remaining = self.total_timesteps - self.model.num_timesteps
        self.pbar = tqdm(total=max(remaining, 0), desc=self.desc, unit="step")

    def _on_step(self):
        self.pbar.update(self.training_env.num_envs)
        return True

    def _on_training_end(self):
        self.pbar.n = self.pbar.total
        self.pbar.refresh()
        self.pbar.close()

# Le celle di training sotto usano TqdmProgressCallback (progress_bar=False
# passato a model.learn) invece dell'opzione integrata progress_bar=True.
# Le celle di valutazione su più episodi usano tqdm direttamente per lo
# stesso motivo: alcune celle (training SAC, training PPO/SAC di Q2)
# possono richiedere diversi minuti con FAST_MODE = False.

RNG_SEED = 42
np.random.seed(RNG_SEED)

MODELS_DIR = "models"
LOGS_DIR = "logs"
FIG_DIR = "figures"
for d in (MODELS_DIR, LOGS_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

plt.rcParams["figure.dpi"] = 110
print("Setup OK — gymnasium, stable-baselines3, ambiente e baseline caricati.")


In [ ]:
# --- Config generale del notebook -----------------------------------------
# Impostare FAST_MODE = True per una passata di fumo veloce (pochi timestep,
# poche run di valutazione) che verifica che tutte le celle girino senza
# errori. Impostare FAST_MODE = False per i valori "di ricerca" consigliati
# nel README del progetto, che richiedono training più lunghi.
# FAST_MODE imposta solo i DEFAULT: TIMESTEPS_Q1 e TIMESTEPS_Q2 sono due
# variabili indipendenti, modificabili separatamente (Q1 allena solo SAC,
# Q2 allena PPO e SAC "a parità di step" quindi condividono un solo valore).
FAST_MODE = True

TIMESTEPS_Q1 = 20_000 if FAST_MODE else 150_000   # training SAC per Q1 (indipendente da Q2)
TIMESTEPS_Q2 = 20_000 if FAST_MODE else 150_000   # training PPO+SAC per Q2 (indipendente da Q1)
N_TEST_EPISODES = 10 if FAST_MODE else 30

print(f"FAST_MODE={FAST_MODE} | TIMESTEPS_Q1={TIMESTEPS_Q1} | "
      f"TIMESTEPS_Q2={TIMESTEPS_Q2} | N_TEST_EPISODES={N_TEST_EPISODES}")


In [ ]:
# Device di training: "auto" (lascia scegliere a Stable-Baselines3: GPU se
# torch.cuda.is_available(), altrimenti CPU), oppure forzare esplicitamente
# "cuda" o "cpu".
#
# NB: con reti piccole come queste (MlpPolicy, [128, 128]) e un singolo
# environment non vettorizzato, la GPU spesso NON è più veloce della CPU —
# il costo di spostare tensori piccoli avanti e indietro tra CPU e GPU ad
# ogni step può superare il guadagno del calcolo stesso (è un caveat noto di
# Stable-Baselines3, non un problema di questo notebook). Vale la pena
# provare entrambi e confrontare i tempi stampati dalle celle di training.
DEVICE = "auto"  # "auto" | "cuda" | "cpu"

cuda_available = torch.cuda.is_available()
effective_device = ("cuda" if cuda_available else "cpu") if DEVICE == "auto" else DEVICE
print(f"DEVICE={DEVICE!r} | GPU (CUDA) disponibile: {cuda_available} | "
      f"device effettivo: {effective_device}")


## 1. Ambiente e funzioni di valutazione comuni

Richiamo rapido dell'ambiente (dettagli completi in `lighting_env.py` /
`README.md`):

- **Episodio** = un giorno simulato, 96 step da 15 minuti.
- **Osservazione** (9 valori normalizzati): ora (sin/cos), lux esterno
  potenziale, lux interno attuale, posizione tapparella, livello LED, lux
  target, stagione, occupazione.
- **Azione** continua `Box(2,)` in `[0,1]`: tapparella e LED, con velocità
  di movimento limitata per step.
- **Reward** (quando occupata): penalità errore di lux, bonus sfruttamento
  luce naturale, penalità abbagliamento, penalità energetica LED, penalità
  di jitter (movimenti bruschi).
- Target di lux, stagione, nuvolosità e occupazione **cambiano ad ogni
  `reset()`**: la policy deve generalizzare, non imparare una routine fissa.

Le funzioni sotto sono condivise dalle tre sezioni Q1/Q2/Q3: eseguono un
episodio con una policy (RL o baseline) e calcolano le metriche di
interesse (errore medio di lux, energia consumata, "jitter" degli
attuatori).


In [ ]:
LED_POWER_WATTS = 24.0            # potenza LED a piena potenza (ipotesi di progetto)
ENERGY_PRICE_EUR_PER_KWH = 0.25   # prezzo indicativo energia


def energy_kwh(led_levels, step_minutes=AdaLux.STEP_MINUTES,
               power_watts=LED_POWER_WATTS):
    '''Energia (kWh) consumata dai LED durante l'episodio.'''
    hours_per_step = step_minutes / 60.0
    return float(sum(l * power_watts / 1000.0 * hours_per_step for l in led_levels))


def actuator_jitter(records):
    '''Movimento medio e massimo per step degli attuatori (tapparella+LED),
    usato in Q3 come proxy di movimenti bruschi.'''
    if len(records) < 2:
        return 0.0, 0.0
    deltas = [
        abs(records[i]["blind_pos"] - records[i - 1]["blind_pos"])
        + abs(records[i]["led_level"] - records[i - 1]["led_level"])
        for i in range(1, len(records))
    ]
    return float(np.mean(deltas)), float(np.max(deltas))


def run_episode(policy, seed, env_cls=AdaLux, is_baseline=False):
    '''Esegue un episodio deterministico con una policy RL (SB3) o con il
    ThresholdController (is_baseline=True) e ritorna la lista di info
    per ogni step (arricchita con il reward).'''
    env = env_cls(seed=seed)
    obs, _ = env.reset(seed=seed)
    if hasattr(policy, "reset"):
        policy.reset()
    done = False
    records = []
    while not done:
        if is_baseline:
            action, _ = policy.predict(obs, env=env)
        else:
            action, _ = policy.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        info["reward"] = reward
        records.append(info)
        done = terminated or truncated
    return records, env.lux_target, env.season_summer


def episode_metrics(records):
    '''Errore medio di lux (solo ore occupate) ed energia (kWh) di un episodio.'''
    occ = [r for r in records if r["occupied"]]
    mean_lux_err = float(np.mean([r["lux_error"] for r in occ])) if occ else 0.0
    mean_blind_pos_occ = float(np.mean([r["blind_pos"] for r in occ])) if occ else 0.0
    kwh = energy_kwh([r["led_level"] for r in records])
    j_mean, j_max = actuator_jitter(records)
    return {
        "mean_lux_error_pct": mean_lux_err * 100.0,
        "energy_kwh": kwh,
        "jitter_mean": j_mean,
        "jitter_max": j_max,
        # frazione [0-1] di quanto la tapparella e' aperta nelle ore occupate:
        # e' la metrica piu' diretta per capire se una policy sfrutta la luce
        # naturale o collassa su "tapparella chiusa + solo LED" (vedi sezione
        # di rigore multi-seed in Q2).
        "mean_blind_pos_occ": mean_blind_pos_occ,
    }


def evaluate_over_seeds(policy, seeds, env_cls=AdaLux, is_baseline=False,
                         label="policy"):
    '''Valuta una policy su piu' episodi/seed di test e ritorna un DataFrame
    con una riga per episodio.'''
    rows = []
    for seed in tqdm(seeds, desc=f"Valutazione {label}", leave=False):
        records, lux_target, season = run_episode(policy, seed, env_cls=env_cls,
                                                    is_baseline=is_baseline)
        m = episode_metrics(records)
        m.update({"seed": seed, "policy": label, "lux_target": lux_target,
                   "season_summer": season})
        rows.append(m)
    return pd.DataFrame(rows)


def plot_day(records, lux_target, season_summer, title_suffix=""):
    '''Riproduce il grafico a 2 pannelli usato per l'ispezione
    qualitativa di un episodio (lux + attuatori nel tempo).'''
    hours = [r["hour"] for r in records]
    indoor_lux = [r["indoor_lux"] for r in records]
    ext_lux = [r["ext_lux"] for r in records]
    blind = [r["blind_pos"] for r in records]
    led = [r["led_level"] for r in records]
    occ = [r["occupied"] for r in records]

    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

    ax = axes[0]
    ax.plot(hours, indoor_lux, label="Lux interno", color="#d97706", linewidth=2)
    ax.plot(hours, ext_lux, label="Lux esterno (potenziale)", color="#94a3b8",
            linestyle="--", linewidth=1)
    ax.axhline(lux_target, color="#16a34a", linestyle=":", label="Target lux")
    _shade_occupancy(ax, hours, occ)
    ax.set_ylabel("Lux")
    ax.set_title(f"{'Estate' if season_summer else 'Inverno/mezza stagione'} — "
                 f"target {lux_target:.0f} lux {title_suffix}")
    ax.legend(loc="upper right", fontsize=8)

    ax = axes[1]
    ax.plot(hours, blind, label="Tapparella (0=chiusa,1=aperta)", color="#2563eb")
    ax.plot(hours, led, label="LED (0=spento,1=max)", color="#f59e0b")
    _shade_occupancy(ax, hours, occ)
    ax.set_ylabel("Livello [0-1]")
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Ora del giorno")
    ax.legend(loc="upper right", fontsize=8)

    fig.tight_layout()
    return fig


def _shade_occupancy(ax, hours, occ):
    start = None
    for i, o in enumerate(occ):
        if o and start is None:
            start = hours[i]
        if (not o or i == len(occ) - 1) and start is not None:
            end = hours[i]
            ax.axvspan(start, end, color="#22c55e", alpha=0.07)
            start = None


def make_monitored_env(seed, log_dir):
    os.makedirs(log_dir, exist_ok=True)
    env = AdaLux(seed=seed)
    env = TimeLimit(env, max_episode_steps=AdaLux.STEPS_PER_DAY)
    env = Monitor(env, filename=os.path.join(log_dir, "monitor"))
    return env


# Seed dei set di test: NON sovrapposti ai seed tipicamente usati in training
# (gli script di training originari usavano seed=42/43), per una valutazione onesta out-of-training-seed.
TEST_SEEDS = list(range(1000, 1000 + N_TEST_EPISODES))
print(f"{len(TEST_SEEDS)} seed di test predisposti: {TEST_SEEDS[:5]}...")


## 2. Q1 — Un agente RL supera un controllore a soglie fisse?

**Baseline**: `ThresholdController` (`baseline_controller.py`) — un
controllore a regole fisse rappresentativo di una logica di building
automation tradizionale:

- tarato su un **setpoint nominale fisso** (500 lux) e **non conosce** il
  target reale richiesto nell'episodio (a differenza dell'agente RL, che lo
  osserva);
- tapparella su pochi livelli discreti scelti da soglie di lux assolute;
- LED con controllo bang-bang e banda morta attorno al setpoint nominale.

**Metodo**: alleniamo (o carichiamo, se già presente) un agente **SAC**
sullo stesso ambiente `AdaLux`, poi confrontiamo SAC e
baseline sullo stesso set di episodi di test (stessi seed → stesse
condizioni meteo/target/occupazione per entrambi).

**Valutazione**: errore medio di lux (ore occupate) ed energia consumata
(kWh) sul set di test.

Il modello viene salvato in `models/sac_q1.zip` (nome uniforme a
`ppo_q2.zip`/`sac_q2.zip` di Q2): con `RETRAIN_Q1 = False` (default) viene
ricaricato da disco se già presente, invece di riallenare da zero. Il
numero di timestep di training è `TIMESTEPS_Q1`, impostabile in modo
indipendente da `TIMESTEPS_Q2` nella cella di configurazione generale.


In [ ]:
RETRAIN_Q1 = False  # True per riallenare da zero invece di caricare il modello salvato
MODEL_Q1_SAC_PATH = os.path.join(MODELS_DIR, "sac_q1")  # nome uniforme a ppo_q2/sac_q2

if RETRAIN_Q1 or not os.path.exists(MODEL_Q1_SAC_PATH + ".zip"):
    print(f"Training SAC per {TIMESTEPS_Q1} timestep...")
    env_q1 = make_monitored_env(seed=RNG_SEED, log_dir=os.path.join(LOGS_DIR, "q1_sac"))
    sac_q1 = SAC(
        "MlpPolicy", env_q1, verbose=0, seed=RNG_SEED, device=DEVICE,
        learning_rate=3e-4, buffer_size=200_000, batch_size=256,
        gamma=0.995, tau=0.005, train_freq=1, gradient_steps=1,
        ent_coef="auto", policy_kwargs=dict(net_arch=[128, 128]),
    )
    t0 = time.time()
    sac_q1.learn(total_timesteps=TIMESTEPS_Q1, log_interval=50,
                 callback=TqdmProgressCallback(TIMESTEPS_Q1, desc="SAC (Q1)"))
    print(f"Training completato in {time.time() - t0:.1f}s")
    sac_q1.save(MODEL_Q1_SAC_PATH)
    print(f"Modello salvato in {MODEL_Q1_SAC_PATH}.zip")
else:
    print(f"Carico il modello SAC già addestrato da {MODEL_Q1_SAC_PATH}.zip")
    sac_q1 = SAC.load(MODEL_Q1_SAC_PATH)

baseline = ThresholdController()
print("Baseline (ThresholdController) pronta.")


In [ ]:
df_baseline = evaluate_over_seeds(baseline, TEST_SEEDS, is_baseline=True, label="Baseline (soglie fisse)")
df_sac_q1 = evaluate_over_seeds(sac_q1, TEST_SEEDS, is_baseline=False, label="SAC")

df_q1 = pd.concat([df_baseline, df_sac_q1], ignore_index=True)
summary_q1 = df_q1.groupby("policy")[["mean_lux_error_pct", "energy_kwh"]].agg(["mean", "std"])
summary_q1


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for ax, col, title, ylabel in [
    (axes[0], "mean_lux_error_pct", "Errore medio di lux (ore occupate)", "Errore [%]"),
    (axes[1], "energy_kwh", "Energia consumata dai LED", "Energia [kWh/giorno]"),
]:
    means = df_q1.groupby("policy")[col].mean()
    stds = df_q1.groupby("policy")[col].std()
    ax.bar(means.index, means.values, yerr=stds.values, capsize=4,
           color=["#94a3b8", "#2563eb"])
    ax.set_title(title)
    ax.set_ylabel(ylabel)

fig.suptitle(f"Q1 — SAC vs baseline a soglie fisse ({len(TEST_SEEDS)} episodi di test)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q1_sac_vs_baseline.png"), dpi=150)
plt.show()


In [ ]:
err_b = df_q1[df_q1.policy.str.contains("Baseline")]["mean_lux_error_pct"].mean()
err_s = df_q1[df_q1.policy == "SAC"]["mean_lux_error_pct"].mean()
kwh_b = df_q1[df_q1.policy.str.contains("Baseline")]["energy_kwh"].mean()
kwh_s = df_q1[df_q1.policy == "SAC"]["energy_kwh"].mean()

print(f"Errore medio di lux — baseline: {err_b:.1f}% | SAC: {err_s:.1f}% "
      f"(riduzione: {(1 - err_s / err_b) * 100:.0f}%)" if err_b > 0 else "")
print(f"Energia media/giorno — baseline: {kwh_b:.3f} kWh | SAC: {kwh_s:.3f} kWh")


### Trade-off comfort luminoso vs energia

I due grafici a barre sopra mostrano errore di lux ed energia
**separatamente**: da soli non dicono se SAC riduce l'errore *a scapito* di
più energia, o se domina la baseline su entrambi gli assi
contemporaneamente. Lo scatter e il confronto appaiato sotto mettono le due
grandezze in relazione diretta, episodio per episodio (stesso seed →
stesse condizioni meteo/target/occupazione per baseline e SAC).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6.4))

colors = {"Baseline (soglie fisse)": "#94a3b8", "SAC": "#2563eb"}
for policy, color in colors.items():
    sub = df_q1[df_q1.policy == policy]
    ax.scatter(sub["energy_kwh"], sub["mean_lux_error_pct"], label=policy,
               color=color, s=60, alpha=0.85, edgecolor="white", linewidth=0.6)

ax.set_xlabel("Energia consumata [kWh/giorno]")
ax.set_ylabel("Errore medio di lux [%]")
fig.suptitle("Q1 — Trade-off comfort luminoso vs energia", fontsize=12, y=0.98)
ax.set_title("un punto = un episodio di test — in basso a sinistra è meglio",
             fontsize=9, color="#555555")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q1_tradeoff_scatter.png"), dpi=150)
plt.show()


In [ ]:
# Confronto appaiato per seed (stesse condizioni per baseline e SAC in ogni
# episodio): su quanti episodi SAC domina la baseline su ENTRAMBI gli assi
# (errore di lux minore E energia minore o uguale, quindi senza alcun
# trade-off sfavorevole), e in quanti invece riduce l'errore ma consuma
# più energia (un vero trade-off comfort/energia).
paired = df_baseline.merge(df_sac_q1, on="seed", suffixes=("_baseline", "_sac"))

dominates = (
    (paired["mean_lux_error_pct_sac"] <= paired["mean_lux_error_pct_baseline"]) &
    (paired["energy_kwh_sac"] <= paired["energy_kwh_baseline"])
)
lower_error_higher_energy = (
    (paired["mean_lux_error_pct_sac"] < paired["mean_lux_error_pct_baseline"]) &
    (paired["energy_kwh_sac"] > paired["energy_kwh_baseline"])
)
delta_energy = paired["energy_kwh_sac"] - paired["energy_kwh_baseline"]
n_total = len(paired)

print(f"SAC domina la baseline (errore minore E energia minore o uguale) in "
      f"{int(dominates.sum())}/{n_total} episodi di test.")
print(f"SAC riduce l'errore ma consuma più energia (vero trade-off) in "
      f"{int(lower_error_higher_energy.sum())}/{n_total} episodi.")
print(f"Delta medio di energia (SAC - baseline): {delta_energy.mean():+.4f} kWh/giorno "
      f"(std: {delta_energy.std():.4f})")


**Come leggere il risultato**: se SAC mostra un errore medio di lux
sensibilmente più basso della baseline (a parità o minor consumo
energetico) sul set di test, la risposta a **Q1** è positiva. Nota che il
vantaggio di SAC qui deriva anche dal fatto che osserva il target reale
dell'episodio (obs. indice 6), mentre la baseline è tarata su un unico
setpoint fisso — è un vantaggio strutturale dell'approccio RL/adattivo, non
solo del training in sé.

Per il trade-off: se la maggioranza degli episodi cade nel quadrante "SAC
domina" (errore minore E energia minore o uguale), il miglioramento di
comfort non ha un costo energetico — è un risultato più forte di un
semplice "in media SAC è migliore su entrambe le metriche", perché regge
episodio per episodio. Se invece emergono episodi con errore minore ma
energia maggiore, quello è un trade-off reale da riportare esplicitamente
(quanti episodi, di quanta energia in più) invece di lasciarlo implicito
nei due grafici a barre separati.

## 3. Q2 — Quale algoritmo converge meglio ed evita ottimi locali?

**Metodo**: alleniamo **PPO** e **SAC** sullo stesso ambiente per lo
**stesso numero di timestep** (`TIMESTEPS_Q2`), stessa architettura di rete
(`[128, 128]`), loggando il reward per episodio con `Monitor` di SB3.

**Valutazione**:
1. curve di reward (media mobile) a confronto durante il training;
2. osservazione qualitativa: un episodio di valutazione deterministico per
   ciascun algoritmo, sullo stesso seed, per vedere se la policy appresa
   sfrutta la luce naturale o collassa su una soluzione sub-ottimale (es.
   tapparella sempre chiusa, come osservato nel README per PPO);
3. **rigore multi-seed**: ripetiamo il training di PPO e SAC su 3 seed
   indipendenti per capire se un comportamento osservato con un solo seed
   (es. l'ottimo locale di PPO) è sistematico o solo un caso isolato di
   quella singola run.

Entrambi i modelli addestrati vengono salvati in `models/ppo_q2.zip` e
`models/sac_q2.zip`: se li riesegui con `RETRAIN_Q2 = False` (default),
il notebook li ricarica da disco invece di riallenarli da zero.


In [ ]:
def train_and_log(algo_name, timesteps, log_dir, seed=RNG_SEED, device=DEVICE,
                   save_path=None, retrain=True):
    '''Allena PPO/SAC loggando il reward per episodio con Monitor in log_dir.
    Se save_path è indicato, il modello addestrato viene salvato su disco
    (save_path + ".zip"). Se retrain=False e in save_path esiste già un
    modello salvato, lo carica invece di riallenare da zero (stesso pattern
    già usato per SAC in Q1) — utile per non dover rifare 150k step di
    training ogni volta che riapri il notebook.'''
    algo_cls = {"PPO": PPO, "SAC": SAC}.get(algo_name)
    if algo_cls is None:
        raise ValueError(algo_name)

    if not retrain and save_path is not None and os.path.exists(save_path + ".zip"):
        print(f"Carico {algo_name} già addestrato da {save_path}.zip "
              f"(retrain=False — le curve di reward sotto restano quelle del training precedente)")
        return algo_cls.load(save_path)

    env = make_monitored_env(seed=seed, log_dir=log_dir)
    common_kwargs = dict(verbose=0, seed=seed, device=device, policy_kwargs=dict(net_arch=[128, 128]))
    if algo_name == "PPO":
        model = PPO("MlpPolicy", env, n_steps=256, batch_size=256, n_epochs=10,
                    gamma=0.995, gae_lambda=0.95, learning_rate=3e-4,
                    ent_coef=0.01, clip_range=0.2, **common_kwargs)
    else:  # SAC
        model = SAC("MlpPolicy", env, learning_rate=3e-4, buffer_size=200_000,
                    batch_size=256, gamma=0.995, tau=0.005, train_freq=1,
                    gradient_steps=1, ent_coef="auto", **common_kwargs)

    t0 = time.time()
    model.learn(total_timesteps=timesteps, log_interval=50,
                callback=TqdmProgressCallback(timesteps, desc=algo_name))
    print(f"{algo_name}: training di {timesteps} step completato in {time.time() - t0:.1f}s")
    if save_path is not None:
        model.save(save_path)
        print(f"Modello salvato in {save_path}.zip")
    return model


RETRAIN_Q2 = False  # True per riallenare da zero anche se i modelli sono già stati salvati in precedenza
LOG_Q2_PPO = os.path.join(LOGS_DIR, "q2_ppo")
LOG_Q2_SAC = os.path.join(LOGS_DIR, "q2_sac")
MODEL_Q2_PPO_PATH = os.path.join(MODELS_DIR, "ppo_q2")
MODEL_Q2_SAC_PATH = os.path.join(MODELS_DIR, "sac_q2")

ppo_q2 = train_and_log("PPO", TIMESTEPS_Q2, LOG_Q2_PPO, seed=RNG_SEED,
                        save_path=MODEL_Q2_PPO_PATH, retrain=RETRAIN_Q2)
sac_q2 = train_and_log("SAC", TIMESTEPS_Q2, LOG_Q2_SAC, seed=RNG_SEED,
                        save_path=MODEL_Q2_SAC_PATH, retrain=RETRAIN_Q2)


In [ ]:
def smoothed_reward_curve(log_dir, window=10):
    x, y = ts2xy(load_results(log_dir), "timesteps")
    if len(y) >= window:
        y_smooth = pd.Series(y).rolling(window, min_periods=1).mean().values
    else:
        y_smooth = y
    return x, y_smooth


fig, ax = plt.subplots(figsize=(9, 5))
for log_dir, label, color in [
    (LOG_Q2_PPO, "PPO", "#dc2626"),
    (LOG_Q2_SAC, "SAC", "#2563eb"),
]:
    x, y = smoothed_reward_curve(log_dir)
    ax.plot(x, y, label=label, color=color)

ax.set_xlabel("Timestep di training")
ax.set_ylabel("Reward per episodio (media mobile)")
ax.set_title(f"Q2 — Curve di reward PPO vs SAC (a parità di {TIMESTEPS_Q2} step)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q2_reward_curves.png"), dpi=150)
plt.show()


### Valutazione quantitativa su più episodi di test

Le curve di reward sopra descrivono il *training*; per confrontare le
policy **già addestrate** in modo più solido di un singolo episodio,
valutiamo PPO e SAC (deterministici) sugli stessi `TEST_SEEDS` usati in Q1
— stesse condizioni meteo/target/occupazione per entrambi, episodio per
episodio.


In [ ]:
df_ppo_q2 = evaluate_over_seeds(ppo_q2, TEST_SEEDS, label="PPO")
df_sac_q2 = evaluate_over_seeds(sac_q2, TEST_SEEDS, label="SAC")

df_q2_eval = pd.concat([df_ppo_q2, df_sac_q2], ignore_index=True)
summary_q2_eval = df_q2_eval.groupby("policy")[
    ["mean_lux_error_pct", "energy_kwh", "jitter_mean", "jitter_max"]
].agg(["mean", "std"])
summary_q2_eval


In [ ]:
# Trade-off comfort/energia, stesso tipo di grafico usato in Q1 — un punto
# per episodio di test, in basso a sinistra è meglio su entrambi gli assi.
colors_q2 = {"PPO": "#dc2626", "SAC": "#2563eb"}

fig, ax = plt.subplots(figsize=(8, 6.4))
for policy, color in colors_q2.items():
    sub = df_q2_eval[df_q2_eval.policy == policy]
    ax.scatter(sub["energy_kwh"], sub["mean_lux_error_pct"], label=policy,
               color=color, s=60, alpha=0.85, edgecolor="white", linewidth=0.6)

ax.set_xlabel("Energia consumata [kWh/giorno]")
ax.set_ylabel("Errore medio di lux [%]")
fig.suptitle("Q2 — Trade-off comfort luminoso vs energia: PPO vs SAC", fontsize=12, y=0.98)
ax.set_title("un punto = un episodio di test — in basso a sinistra è meglio",
             fontsize=9, color="#555555")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q2_tradeoff_scatter.png"), dpi=150)
plt.show()


In [ ]:
# Energia consumata episodio per episodio (un punto/seed di test per algoritmo).
fig, ax = plt.subplots(figsize=(9, 4.5))
for policy, color in colors_q2.items():
    sub = df_q2_eval[df_q2_eval.policy == policy].sort_values("seed")
    ax.plot(range(len(sub)), sub["energy_kwh"].values, marker="o", label=policy, color=color)

ax.set_xlabel("Episodio di test (ordinati per seed)")
ax.set_ylabel("Energia [kWh/giorno]")
ax.set_title("Q2 — Energia consumata per episodio di test: PPO vs SAC")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q2_energy_per_episode.png"), dpi=150)
plt.show()


In [ ]:
# Comfort (errore di lux) episodio per episodio, stessa impostazione del
# grafico dell'energia sopra — utile per vedere se un algoritmo è
# sistematicamente peggiore o solo su alcuni episodi/condizioni specifiche.
fig, ax = plt.subplots(figsize=(9, 4.5))
for policy, color in colors_q2.items():
    sub = df_q2_eval[df_q2_eval.policy == policy].sort_values("seed")
    ax.plot(range(len(sub)), sub["mean_lux_error_pct"].values, marker="o", label=policy, color=color)

ax.set_xlabel("Episodio di test (ordinati per seed)")
ax.set_ylabel("Errore medio di lux [%]")
ax.set_title("Q2 — Comfort (errore di lux) per episodio di test: PPO vs SAC")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q2_comfort_per_episode.png"), dpi=150)
plt.show()


In [ ]:
# Jitter degli attuatori: quanto si muovono bruscamente tapparella/LED da
# uno step all'altro. Una policy che ha imparato una soluzione "comoda" ma
# poco raffinata (es. aggiustamenti continui e bruschi invece di un
# controllo fluido) tende ad avere un jitter medio più alto — è un
# indicatore quantitativo aggiuntivo, oltre al reward, del "quanto bene"
# ciascun algoritmo ha davvero imparato a controllare l'ambiente.
means = df_q2_eval.groupby("policy")["jitter_mean"].mean()
stds = df_q2_eval.groupby("policy")["jitter_mean"].std()

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(means.index, means.values, yerr=stds.values, capsize=4,
       color=[colors_q2[p] for p in means.index])
ax.set_ylabel("Jitter medio attuatori (Δposizione/step)")
ax.set_title("Q2 — Movimenti bruschi degli attuatori: PPO vs SAC")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q2_jitter.png"), dpi=150)
plt.show()


**Come leggere questi risultati**: il grafico trade-off e le due serie
per episodio danno un quadro più solido di un singolo episodio qualitativo
— se SAC è sistematicamente nel quadrante basso-sinistra (errore ed
energia entrambi minori) su quasi tutti i `TEST_SEEDS`, il vantaggio non è
un caso isolato. Il grafico del jitter aggiunge un'altra dimensione: una
policy bloccata su un ottimo locale spesso non è solo "peggiore in media",
ma anche meno fluida nel controllo (aggiustamenti bruschi e ripetuti)
rispetto a una policy che ha davvero imparato a bilanciare tapparella e
LED.


In [ ]:
# Osservazione qualitativa: stesso seed di valutazione per entrambi gli algoritmi
QUAL_SEED = 67
records_ppo, target_ppo, season_ppo = run_episode(ppo_q2, QUAL_SEED)
records_sac, target_sac, season_sac = run_episode(sac_q2, QUAL_SEED)

fig_ppo = plot_day(records_ppo, target_ppo, season_ppo, title_suffix="— PPO")
fig_ppo.savefig(os.path.join(FIG_DIR, "q2_ppo_day.png"), dpi=150)
plt.show()

fig_sac = plot_day(records_sac, target_sac, season_sac, title_suffix="— SAC")
fig_sac.savefig(os.path.join(FIG_DIR, "q2_sac_day.png"), dpi=150)
plt.show()

m_ppo = episode_metrics(records_ppo)
m_sac = episode_metrics(records_sac)
pd.DataFrame([{"policy": "PPO", **m_ppo}, {"policy": "SAC", **m_sac}])


**Come leggere il risultato**: oltre al confronto quantitativo delle
curve (chi converge più in fretta e a un livello di reward più alto), il
grafico "qualitativo" mostra se la tapparella viene effettivamente
utilizzata: una policy che la mantiene sempre chiusa e delega tutto ai LED
(come osservato in una prima versione PPO, si veda il `README.md`) è un
sintomo di **ottimo locale** — ha imparato una soluzione "comoda" invece di
scoprire che aprire la tapparella riduce ulteriormente errore ed energia.

### Rigore multi-seed: è un ottimo locale sistematico o un singolo seed sfortunato?

Il confronto sopra usa **un solo training seed** (`RNG_SEED`) per PPO e
SAC: non basta a distinguere un comportamento sistematico dell'algoritmo su
questo ambiente/reward da una singola run che è "andata male" per caso —
esattamente il punto sollevato nella revisione della proposta.

Alleniamo quindi PPO e SAC da zero su **3 training seed indipendenti**
(`SEEDS_Q2_MULTI`), sempre con il numero di timestep "di ricerca"
(`TIMESTEPS_Q2_MULTISEED = 150_000`), **indipendentemente da `FAST_MODE`**:
misurare la robustezza della convergenza con un training abbreviato non
avrebbe senso, quindi qui si usa sempre la modalità completa/lenta, anche
se il resto del notebook gira in `FAST_MODE = True`.

Per ciascuno dei 6 modelli (3 seed × 2 algoritmi) valutiamo, sugli stessi
`TEST_SEEDS` usati sopra:

- errore medio di lux ed energia (come già visto);
- **uso medio della tapparella nelle ore occupate** (`mean_blind_pos_occ`,
  0 = sempre chiusa, 1 = sempre aperta): è la metrica più diretta per
  capire se una policy sfrutta la luce naturale o collassa su "tapparella
  chiusa + solo LED", indipendentemente dal suo errore di lux o energia
  assoluti;
- il reward finale di training (media mobile sugli ultimi punti loggati),
  come proxy sintetico di "quanto bene converge".

**Nota sui tempi**: questo richiede fino a 6 training completi da 150k
step — con reti piccole su CPU può richiedere comunque un'ora o più in
totale. Come per le altre celle di training, ogni modello viene salvato
singolarmente (`models/ppo_q2_seed{{seed}}.zip`,
`models/sac_q2_seed{{seed}}.zip`) e ricaricato da disco se già presente
(`RETRAIN_Q2_MULTISEED = False`), così un'interruzione a metà non fa
perdere il lavoro già fatto.


In [ ]:
SEEDS_Q2_MULTI = [1, 2, 3]
# Sempre "modalita' lenta": qui si misura la robustezza della convergenza,
# quindi non ha senso legarla a FAST_MODE come TIMESTEPS_Q2.
TIMESTEPS_Q2_MULTISEED = 150_000
RETRAIN_Q2_MULTISEED = False  # True per riallenare da zero anche i modelli gia' salvati

models_q2_multi = {}
for seed in tqdm(SEEDS_Q2_MULTI, desc="Seed di training Q2 multi-seed"):
    for algo_name in ("PPO", "SAC"):
        log_dir = os.path.join(LOGS_DIR, f"q2_{algo_name.lower()}_seed{seed}")
        save_path = os.path.join(MODELS_DIR, f"{algo_name.lower()}_q2_seed{seed}")
        model = train_and_log(algo_name, TIMESTEPS_Q2_MULTISEED, log_dir, seed=seed,
                               save_path=save_path, retrain=RETRAIN_Q2_MULTISEED)
        models_q2_multi[(algo_name, seed)] = (model, log_dir)

print(f"{len(models_q2_multi)} modelli pronti ({len(SEEDS_Q2_MULTI)} seed x 2 algoritmi).")


In [ ]:
rows_multi = []
for (algo_name, seed), (model, log_dir) in tqdm(list(models_q2_multi.items()),
                                                  desc="Valutazione multi-seed"):
    df_eval = evaluate_over_seeds(model, TEST_SEEDS, label=f"{algo_name}_seed{seed}")
    _, final_y = smoothed_reward_curve(log_dir)
    final_reward = float(np.mean(final_y[-5:])) if len(final_y) else float("nan")
    rows_multi.append({
        "algo": algo_name,
        "train_seed": seed,
        "mean_lux_error_pct": df_eval["mean_lux_error_pct"].mean(),
        "energy_kwh": df_eval["energy_kwh"].mean(),
        "mean_blind_pos_occ": df_eval["mean_blind_pos_occ"].mean(),
        "final_reward": final_reward,
    })

df_q2_multi = pd.DataFrame(rows_multi)
df_q2_multi


In [ ]:
# Un punto per training seed (5 per algoritmo): il trattino orizzontale e' la
# media sui 3 seed. Mostra se il comportamento osservato con un solo seed
# (RNG_SEED) e' sistematico (poca dispersione tra i pallini) o dipende dal
# seed (pallini molto sparsi).
metrics_multi = [
    ("mean_blind_pos_occ", "Uso medio tapparella\n(ore occupate) [0-1]"),
    ("mean_lux_error_pct", "Errore medio di lux [%]"),
    ("energy_kwh", "Energia [kWh/giorno]"),
    ("final_reward", "Reward finale\n(media mobile)"),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.6))
for ax, (col, ylabel) in zip(axes, metrics_multi):
    for algo_name, color in colors_q2.items():
        sub = df_q2_multi[df_q2_multi.algo == algo_name]
        ax.scatter([algo_name] * len(sub), sub[col], color=color, s=70, alpha=0.85,
                   edgecolor="white", linewidth=0.6, zorder=3)
        ax.scatter([algo_name], [sub[col].mean()], color=color, marker="_", s=1200,
                   linewidth=3, zorder=4)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlim(-0.5, 1.5)

fig.suptitle(f"Q2 — Rigore multi-seed: PPO vs SAC su {len(SEEDS_Q2_MULTI)} training seed "
             f"indipendenti ({TIMESTEPS_Q2_MULTISEED} step ciascuno)", fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q2_multiseed_summary.png"), dpi=150, bbox_inches="tight")
plt.show()


**Come leggere il risultato**: ogni pallino è un training seed
indipendente, il trattino orizzontale la media sui 3 seed. Se i 3 pallini
PPO su "uso medio tapparella" sono tutti vicini a zero (e sistematicamente
sotto quelli SAC su tutti e 5 i seed), l'ottimo locale osservato con
`RNG_SEED=42` **non è un caso isolato**: è un comportamento sistematico di
PPO su questo ambiente/reward. Se invece i risultati PPO sono molto
dispersi da seed a seed (alcuni aprono la tapparella, altri no), la
conclusione "PPO evita/non evita ottimi locali" va sfumata — dipende dal
seed, il tipo di generalizzazione azzardata che la revisione della
proposta segnalava esplicitamente su Q2.


## 4. Q3 — La policy resta affidabile fuori distribuzione (OOD)?

**Metodo**: definiamo `ExtremeAdaLux`, una variante
dell'ambiente che genera condizioni **fuori dal range visto in training**:

- target di lux in `[900, 1400]` (il training usa `[300, 750]`);
- nuvolosità quasi sempre agli estremi (foschia densa o cielo sereno
  estremo) con variazioni più rapide e ampie;
- fascia di occupazione molto più ampia (5:00–22:00 contro 7-10/17-20 del
  training).

**Valutazione**: confrontiamo la policy SAC (quella di Q1) su episodi
"normali" vs "estremi" (stessi seed per le due condizioni), guardando
errore di lux, energia e il **jitter degli attuatori** (movimento medio e
massimo per step) come proxy di movimenti bruschi.


In [ ]:
import numpy as np

class ExtremeAdaLux(AdaLux):
    # Variante dell'ambiente con condizioni fuori dal range di training,
    # per testare la generalizzazione OOD della policy (Q3).

    EXTREME_LUX_TARGET_RANGE = (900.0, 1400.0)

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        # target di lux ben oltre LUX_TARGET_RANGE visto in training
        self.lux_target = float(self._rng.uniform(*self.EXTREME_LUX_TARGET_RANGE))
        # nuvolosità agli estremi (foschia densa o cielo sereno estremo)
        if self._rng.random() < 0.5:
            self.cloud_factor = float(self._rng.uniform(0.05, 0.15))
        else:
            self.cloud_factor = float(self._rng.uniform(0.95, 1.0))
        # occupazione molto più estesa del range di training
        self.occupancy_start = 5
        self.occupancy_end = 22
        obs = self._build_obs()
        return obs, info

    def _update_cloud_factor(self):
        # random walk più ampio/rapido: meteo instabile, non visto in training
        step = self._rng.normal(0, 0.12)
        self.cloud_factor = float(np.clip(self.cloud_factor + step, 0.02, 1.0))


print("ExtremeAdaLux definito.")


In [ ]:
df_normal = evaluate_over_seeds(sac_q1, TEST_SEEDS, env_cls=AdaLux, label="Normale")
df_extreme = evaluate_over_seeds(sac_q1, TEST_SEEDS, env_cls=ExtremeAdaLux, label="Estremo (OOD)")

df_q3 = pd.concat([df_normal, df_extreme], ignore_index=True)
summary_q3 = df_q3.groupby("policy")[["mean_lux_error_pct", "energy_kwh", "jitter_mean", "jitter_max"]].agg(["mean", "std"])
summary_q3


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

panels = [
    ("mean_lux_error_pct", "Errore medio di lux", "Errore [%]"),
    ("energy_kwh", "Energia consumata", "Energia [kWh/giorno]"),
    ("jitter_mean", "Jitter medio attuatori", "Δposizione media/step"),
]
colors = ["#16a34a", "#dc2626"]
for ax, (col, title, ylabel) in zip(axes, panels):
    means = df_q3.groupby("policy")[col].mean()
    stds = df_q3.groupby("policy")[col].std()
    ax.bar(means.index, means.values, yerr=stds.values, capsize=4, color=colors)
    ax.set_title(title)
    ax.set_ylabel(ylabel)

fig.suptitle(f"Q3 — SAC: condizioni normali vs estreme ({len(TEST_SEEDS)} episodi di test)")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "q3_normal_vs_extreme.png"), dpi=150)
plt.show()


In [ ]:
# Ispezione qualitativa di un singolo episodio estremo: si notano movimenti
# bruschi/oscillazioni degli attuatori o saturazione delle azioni?
records_extreme, target_extreme, season_extreme = run_episode(sac_q1, QUAL_SEED, env_cls=ExtremeAdaLux)
fig_extreme = plot_day(records_extreme, target_extreme, season_extreme, title_suffix="— SAC, condizioni estreme")
fig_extreme.savefig(os.path.join(FIG_DIR, "q3_extreme_day.png"), dpi=150)
plt.show()

episode_metrics(records_extreme)


**Come leggere il risultato**: un aumento marcato dell'errore medio di
lux e/o del jitter nelle condizioni estreme rispetto a quelle normali
indica che la policy **non generalizza bene fuori distribuzione** — atteso,
dato che né il target né la nuvolosità/occupazione estremi rientrano nel
range visto in training. Il grafico qualitativo aiuta a capire *come*
fallisce: saturazione delle azioni (tapparella/LED sempre a 0 o 1),
oscillazioni continue, o errore di lux persistente senza reazione.

## 5. Conclusioni

Riepilogo automatico dei risultati calcolati sopra (da rileggere/commentare
dopo un run con `FAST_MODE = False` e i timestep di training consigliati
nel `README.md`, per risultati non "smoke-test").


In [ ]:
print("=== Q1 — SAC vs baseline a soglie fisse ===")
print(summary_q1)
print()
print("=== Q2 — Reward finale (media mobile ultimo punto) PPO vs SAC (seed singolo) ===")
for log_dir, label in [(LOG_Q2_PPO, "PPO"), (LOG_Q2_SAC, "SAC")]:
    x, y = smoothed_reward_curve(log_dir)
    print(f"{label}: reward finale (media mobile) = {y[-1]:.2f}  (ultimo punto a {x[-1]} step)")
print()
print(f"=== Q2 — Rigore multi-seed ({len(SEEDS_Q2_MULTI)} seed, {TIMESTEPS_Q2_MULTISEED} step) ===")
print(df_q2_multi.groupby("algo")[["mean_lux_error_pct", "energy_kwh", "mean_blind_pos_occ", "final_reward"]]
      .agg(["mean", "std"]))
print()
print("=== Q3 — SAC: normale vs estremo ===")
print(summary_q3)


Da completare con l'interpretazione dei risultati numerici e grafici
ottenuti eseguendo il notebook con `FAST_MODE = False`:

- **Q1**: l'agente RL supera il controllore a soglie fisse? Di quanto, su
  errore di lux ed energia?
- **Q2**: quale algoritmo converge più velocemente/più in alto? Ci sono
  segnali di ottimo locale (es. tapparella inutilizzata) in uno dei due?
  Il rigore multi-seed conferma che è un comportamento sistematico, o è
  legato al seed di training?
- **Q3**: quanto si degrada la policy fuori distribuzione? I movimenti
  degli attuatori restano ragionevoli o diventano bruschi/instabili?
